# Gamma 校正（Gamma Correction）手写实现

## 一、什么是 Gamma 校正？

### 1. 定义
Gamma 校正是一种**非线性**的图像调整技术，用于改变图像的亮度和对比度。它基于**幂函数**对像素值进行变换。

### 2. 为什么需要 Gamma 校正？
- **人眼视觉特性**：人眼对光线强度的感知是**非线性**的（对数关系），而显示设备（如显示器）的光强输出是**线性**的。Gamma 校正可以使图像的显示效果更符合人眼的视觉感知。
- **显示器校准**：不同显示器有不同的 Gamma 响应曲线，需要通过 Gamma 校正来补偿。
- **图像增强**：可以用来提亮暗部细节或压缩过亮部分。

### 3. 数学原理

公式：`output = input^(1/gamma)`

其中：
- `input`：归一化后的像素值（范围 0~1）
- `gamma`：控制曲线形状的参数
- `output`：校正后的像素值

### 4. Gamma 值的影响
| Gamma 值 | 效果 | 说明 |
|-----------|------|------|
| gamma > 1 | 图像变亮 | 暗部被提亮，提升暗部细节 |
| gamma < 1 | 图像变暗 | 亮部被压缩 |
| gamma = 1 | 无变化 | 恒等变换 |
| gamma = 2.2 | 典型值 | CRT 显示器的典型 gamma 值 |

### 5. 应用场景
- 显示器色彩校准
- 暗部图像增强（如夜景、室内照片）
- 数字摄影后期处理
- 医学图像处理


## 二、实现要求

> **要求**：使用 Python 调用 OpenCV 的图像读取/保存函数，对彩色图像进行 Gamma 校正。除 OpenCV 的读写函数外，其余代码全部手写，不能调用其他函数库。

这意味着：
- 可以使用 `cv2.imread()` 读取图像
- 可以使用 `cv2.imwrite()` 保存图像
- 不能使用 `cv2.LUT()` 等 OpenCV 的图像处理函数
- 不能使用 `numpy.power()` 等向量化运算
- 不能使用 `PIL`、`skimage` 等其他库
- 可以使用 Python 基本语法（循环、条件、数学运算）


## 三、代码实现

### Step 1：导入 OpenCV 库

我们只导入 `cv2` 模块，用于图像的读取和保存。

In [ ]:
import cv2  # 仅用于图像的读取和保存

print("OpenCV 版本:", cv2.__version__)

### Step 2：手写 Gamma 校正函数

核心实现思路：
1. 遍历图像的每一个像素点（三重循环）
2. 对每个像素的每个通道（B、G、R）分别处理
3. 将像素值从 [0, 255] 归一化到 [0, 1]
4. 应用 Gamma 公式：`output = input^(1/gamma)`
5. 将结果反归一化回 [0, 255]
6. 存储到新图像中

In [ ]:
def gamma_correction_manual(image, gamma):
    """
    手写实现 Gamma 校正
    
    参数:
        image: 输入图像（OpenCV 读取的图像，BGR 格式，uint8 类型）
        gamma: Gamma 值（float）
               - gamma > 1: 提亮暗部
               - gamma < 1: 压暗暗部  
               - gamma = 1:  无变化
    
    返回:
        校正后的图像（uint8 类型）
    """
    
    # 获取图像的高度、宽度和通道数
    # OpenCV 读取的彩色图像形状为 (height, width, 3)
    # 通道顺序为 BGR（注意不是 RGB）
    height, width, channels = image.shape
    
    # 创建结果图像副本，使用 float32 以支持小数运算
    result = image.copy().astype('float32')
    
    # 计算 gamma 的倒数：1/gamma
    # 公式: output = input^(1/gamma)
    # 当 gamma>1 时，1/gamma<1，使暗值变大（提亮）
    # 当 gamma<1 时，1/gamma>1，使暗值变小（压暗）
    inv_gamma = 1.0 / gamma
    
    # ===== 核心：逐像素进行 Gamma 变换 =====
    
    for y in range(height):          # 遍历每一行
        for x in range(width):      # 遍历每一列
            for c in range(channels):  # 遍历 B/G/R 三个通道
                
                # Step 1: 获取当前像素值 (0~255)
                pixel_value = result[y, x, c]
                
                # Step 2: 归一化到 [0, 1] 区间
                normalized = pixel_value / 255.0
                
                # Step 3: 应用 Gamma 公式
                # output = input^(1/gamma)
                gamma_corrected = normalized ** inv_gamma
                
                # Step 4: 反归一化回 [0, 255]
                result[y, x, c] = gamma_corrected * 255.0
    
    # 转换回 uint8 类型（OpenCV 图像标准格式）
    result = result.astype('uint8')
    
    return result

### Step 3：读取图像

使用 `cv2.imread()` 读取彩色图像。

In [ ]:
# ==================== 配置参数 ====================

# 输入图像路径（可修改为自己的图像）
input_image_path = "test_image.jpg"

# 输出图像路径
output_image_path = "gamma_corrected.jpg"

# Gamma 值（可调整）
gamma_value = 2.2  # 提亮暗部

# ==================== 读取图像 ====================
print("正在读取图像...")

# cv2.imread 参数说明:
# 参数1: 图像文件路径
# 参数2: 读取模式
#   - cv2.IMREAD_COLOR (1):     彩色图像（BGR格式）
#   - cv2.IMREAD_GRAYSCALE (0): 灰度图像
#   - cv2.IMREAD_UNCHANGED (-1): 包含alpha通道
img = cv2.imread(input_image_path, cv2.IMREAD_COLOR)

# 检查图像是否成功读取
if img is None:
    print(f"错误：无法读取图像 '{input_image_path}'")
    print("请检查图像文件路径是否正确")
else:
    # 打印图像基本信息
    h, w, ch = img.shape
    print(f"图像读取成功！")
    print(f"  - 尺寸: {w} x {h}")
    print(f"  - 通道数: {ch} (BGR格式)")
    print(f"  - 数据类型: {img.dtype}")
    print(f"  - 像素值范围: [{img.min()}, {img.max()}]")

### Step 4：执行 Gamma 校正

In [ ]:
if img is not None:
    print(f"\n正在进行 Gamma 校正 (gamma={gamma_value})...")
    print("（逐像素处理，对于大图像可能需要一些时间）\n")
    
    # 调用手写的 gamma 校正函数
    corrected_img = gamma_correction_manual(img, gamma_value)
    
    # 打印校正后图像的信息
    print(f"校正完成！")
    print(f"  - 校正后像素值范围: [{corrected_img.min()}, {corrected_img.max()}]")
    print(f"  - 校正后平均值: {corrected_img.mean():.2f}")

### Step 5：保存结果图像

In [ ]:
if img is not None:
    print(f"\n正在保存结果图像到 '{output_image_path}'...")
    
    # cv2.imwrite 参数说明:
    # 参数1: 保存路径（扩展名决定格式：.jpg, .png, .bmp 等）
    # 参数2: 要保存的图像数组（必须是 uint8 类型）
    success = cv2.imwrite(output_image_path, corrected_img)
    
    if success:
        print(f"保存成功！结果已保存到: {output_image_path}")
    else:
        print(f"保存失败！请检查文件路径和格式。")

## 四、代码总结

### 完整流程
```
读取图像 -> 逐像素遍历 -> 归一化 -> Gamma公式 -> 反归一化 -> 保存图像
```

### 关键代码段解析

| 步骤 | 代码 | 说明 |
|------|------|------|
| 1. 读取 | `cv2.imread()` | 使用 OpenCV 读取彩色图像 |
| 2. 获取维度 | `image.shape` | 获取图像高、宽、通道数 |
| 3. 归一化 | `pixel / 255.0` | 将 [0,255] 映射到 [0,1] |
| 4. Gamma 变换 | `normal ** inv_gamma` | 幂函数运算 |
| 5. 反归一化 | `result * 255.0` | 将 [0,1] 映射回 [0,255] |
| 6. 保存 | `cv2.imwrite()` | 使用 OpenCV 保存结果 |

### 注意事项
1. **OpenCV 通道顺序是 BGR，不是 RGB**：这与大多数图像处理库不同
2. **像素值范围是 [0, 255]**：uint8 类型，需要归一化到 [0, 1] 做运算
3. **逐像素处理较慢**：对于大图像，三重循环会比较耗时
4. **JPEG 有损压缩**：保存为 JPEG 格式时会引入微小误差
5. **边界情况**：gamma=1 时图像不变；gamma 值过大会导致严重失真


## 五、验证实验

让我们通过一个具体像素来验证 Gamma 校正的正确性：

In [ ]:
# 读取原图和校正后的图像进行对比
original = cv2.imread(input_image_path)
corrected = cv2.imread(output_image_path)

if original is not None and corrected is not None:
    print("=== 原图 vs 校正后对比 ===\n")
    
    # 取一个像素进行公式验证
    h, w = 100, 200  # 选择图像中间偏右的位置
    
    print(f"验证位置: 行={h}, 列={w}")
    print(f"{'通道':<8} {'原值':<8} {'归一化':<10} {'公式结果':<10} {'实际值':<8}")
    print("-" * 55)
    
    for c, name in enumerate(['B', 'G', 'R']):
        orig_val = original[h, w, c]
        norm_val = orig_val / 255.0
        formula_val = (norm_val ** (1.0/gamma_value)) * 255.0
        actual_val = corrected[h, w, c]
        
        print(f"{name:<8} {orig_val:<8} {norm_val:<10.4f} {formula_val:<10.1f} {actual_val:<8}")
    
    # 整体统计
    print(f"\n原图统计:")
    print(f"  平均值: {original.mean():.2f}")
    print(f"  最小值: {original.min()}, 最大值: {original.max()}")
    
    print(f"\n校正后统计:")
    print(f"  平均值: {corrected.mean():.2f}")
    print(f"  最小值: {corrected.min()}, 最大值: {corrected.max()}")
    
    # 判断效果
    if corrected.mean() > original.mean():
        print(f"\n验证通过！gamma={gamma_value} 使图像整体变亮")
    else:
        print(f"\n验证通过！gamma={gamma_value} 使图像整体变暗")
else:
    print("无法读取图像进行验证")

## 六、扩展练习

尝试修改 `gamma_value` 参数，观察不同 gamma 值的效果：

1. `gamma = 0.5`：使图像变暗
2. `gamma = 1.0`：无变化
3. `gamma = 3.0`：大幅提亮暗部

### 思考题
1. 如果要对灰度图像进行 Gamma 校正，代码需要做哪些修改？
2. 为什么选择 `output = input^(1/gamma)` 而不是 `output = input^gamma`？
3. 如何加速逐像素处理？（提示：可以使用 numpy 的向量化运算，但本题不允许）